In [1]:
import geopandas as gpd
from shapely.geometry import LineString, Point

In [2]:
import geopandas as gpd
from shapely.geometry import MultiLineString, LineString, Point

# Load the GeoPackage layer
layer_path = "data/Foundation_Data.gpkg"
layer_name = "edges"  # Replace with the name of your layer
edges_gdf = gpd.read_file(layer_path, layer=layer_name)

# Decompose MultiLineString into individual LineStrings
def explode_multilinestring(geometry):
    """Decomposes MultiLineString into individual LineStrings."""
    if isinstance(geometry, MultiLineString):
        return list(geometry.geoms)
    elif isinstance(geometry, LineString):
        return [geometry]
    return []

# Create a new GeoDataFrame with decomposed LineStrings
lines = []
for _, row in edges_gdf.iterrows():
    for line in explode_multilinestring(row.geometry):
        lines.append({'id': row['id'], 'geometry': line})

decomposed_gdf = gpd.GeoDataFrame(lines, crs=edges_gdf.crs)

# Extract start and end points of each LineString
decomposed_gdf['start_point'] = decomposed_gdf.geometry.apply(lambda geom: Point(geom.coords[0]))
decomposed_gdf['end_point'] = decomposed_gdf.geometry.apply(lambda geom: Point(geom.coords[-1]))

# Combine all start and end points into a single GeoDataFrame
points_gdf = gpd.GeoDataFrame(
    data={
        'edge_id': decomposed_gdf['id'].tolist() * 2,  # Each edge contributes two points
        'type': ['start'] * len(decomposed_gdf) + ['end'] * len(decomposed_gdf),
    },
    geometry=list(decomposed_gdf['start_point']) + list(decomposed_gdf['end_point']),
    crs=decomposed_gdf.crs
)

# Count occurrences of each point
points_counts = points_gdf.groupby('geometry').size()

# Identify points that appear only once (breaks in the line)
break_points = points_counts[points_counts == 1].index

# Find edges corresponding to break points
break_edges = decomposed_gdf[
    decomposed_gdf.apply(lambda row: row['start_point'] in break_points or row['end_point'] in break_points, axis=1)
]

# Print or return the IDs of the broken edges
broken_edge_ids = break_edges['id'].unique().tolist()
print("Edges with breaks in the coastline:", broken_edge_ids)


Edges with breaks in the coastline: ['edge_0', 'edge_29', 'edge_30', 'edge_2605', 'edge_2606', 'edge_2697', 'edge_2698', 'edge_2805', 'edge_2806', 'edge_2988', 'edge_2989', 'edge_3622']


In [19]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# Load the GeoPackage layer
input_gpkg = "Foundation_Data.gpkg"  # Replace with your file path
edge_layer_name = "edges"    # Replace with your edge layer name
edges = gpd.read_file(input_gpkg, layer=edge_layer_name)

# Create a DataFrame to store unique nodes
node_set = {}

# Function to round coordinates to a fixed number of decimals
def round_coords(coords, decimals=5):
    return tuple(round(c, decimals) for c in coords)

# Function to get or create a node ID for a point
def get_node_id(point, node_counter, decimals=5):
    rounded_point = round_coords(point, decimals)
    if rounded_point not in node_set:
        node_id = f"node_{node_counter[0]}"
        node_set[rounded_point] = node_id
        node_counter[0] += 1
        return node_id
    return node_set[rounded_point]

# Initialize node counter
node_counter = [1]

# Add from_node and to_node fields to edges
edges["from_node"] = ""
edges["to_node"] = ""

for index, row in edges.iterrows():
    line = row.geometry
    if line is not None and line.geom_type == "LineString":
        # Extract the start and end points
        start_point = tuple(line.coords[0])
        end_point = tuple(line.coords[-1])

        # Assign node IDs
        start_node_id = get_node_id(start_point, node_counter)
        end_node_id = get_node_id(end_point, node_counter)

        # Add IDs to the edge
        edges.at[index, "from_node"] = start_node_id
        edges.at[index, "to_node"] = end_node_id

# Create node GeoDataFrame
nodes = gpd.GeoDataFrame(
    [{"node_id": node_id, "geometry": Point(coords)} for coords, node_id in node_set.items()],
    crs=edges.crs
)

# Save the updated layers to a new GeoPackage
output_gpkg = "Foundation_Data.gpkg"
edges.to_file(output_gpkg, layer="edges", driver="GPKG")
nodes.to_file(output_gpkg, layer="nodes", driver="GPKG")

print("Node and edge layers have been successfully created and saved.")


Node and edge layers have been successfully created and saved.


In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon

def clean_multipolygons(input_gpkg, layer_name):
    """
    Reads a GeoPackage layer, keeps only the largest Polygon in each MultiPolygon,
    and overwrites the input file with the updated geometries.
    
    Parameters:
        input_gpkg (str): Path to the GeoPackage file.
        layer_name (str): Name of the layer to process.
    """
    # Load the GeoPackage
    gdf = gpd.read_file(input_gpkg, layer=layer_name)
    
    # Keep the largest polygon for each geometry
    gdf['geometry'] = gdf['geometry'].apply(
        lambda geom: max(geom.geoms, key=lambda poly: poly.area) if isinstance(geom, MultiPolygon) else geom
    )
    
    # Save back to the same file
    gdf.to_file(input_gpkg, layer=layer_name, driver="GPKG")
    print(f"Updated {input_gpkg}, layer '{layer_name}' with largest polygons.")

# Example usage
clean_multipolygons("jamaica_polygon.gpkg", "jam")


In [ ]:
import geopandas as gpd
from shapely.ops import polygonize
from shapely.ops import linemerge
from shapely.geometry import LineString, Polygon

import geopandas as gpd

def order_edges_around_island(edge_layer):
    """
    Order edges around an island, starting with edge_0 and following the
    connections based on from_id and to_id.

    Parameters:
    - edge_layer (GeoDataFrame): GeoDataFrame containing edges with 'id', 'from_id', and 'to_id' columns

    Returns:
    - List[str]: List of edge IDs in the order they are traced around the island
    """
    # Initialize the ordered list with the first edge
    ordered_edges = []

    # Create a lookup dictionary for edges by their 'from_id'
    edges_by_from_id = {
        edge['from_id']: edge
        for _, edge in edge_layer.iterrows()
    }

    # Start with edge_0
    current_edge = edge_layer[edge_layer['id'] == 'edge_0'].iloc[0]
    ordered_edges.append(current_edge['id'])

    # Track the starting node for circular detection
    start_node = current_edge['from_id']

    # Follow the edges until we complete the loop
    while True:
        # Get the next edge based on the current edge's to_id
        next_from_id = current_edge['to_id']

        # Break the loop if we return to the starting edge
        if next_from_id == start_node:
            break

        # Find the next edge using the lookup dictionary
        current_edge = edges_by_from_id[next_from_id]
        ordered_edges.append(current_edge['id'])

    return ordered_edges

def filter_and_reorder_edges(edge_layer, ordered_edge_ids):
    """
    Filter and reorder the edge layer GeoDataFrame based on the given ordered edge IDs.

    Parameters:
    - edge_layer (GeoDataFrame): GeoDataFrame containing edges with an 'id' column.
    - ordered_edge_ids (List[str]): List of edge IDs in the desired order.

    Returns:
    - GeoDataFrame: A new GeoDataFrame filtered and ordered by `ordered_edge_ids`.
    """
    # Create a dictionary for quick lookup of edges by ID
    edges_by_id = {edge['id']: edge for _, edge in edge_layer.iterrows()}

    # Build the reordered list of edges
    reordered_edges = [edges_by_id[edge_id] for edge_id in ordered_edge_ids if edge_id in edges_by_id]

    # Create a new GeoDataFrame with the reordered edges
    reordered_edge_layer = gpd.GeoDataFrame(reordered_edges, columns=edge_layer.columns, crs=edge_layer.crs)

    return reordered_edge_layer

def create_polygon_from_edges(reordered_edge_layer):
    """
    Create a polygon from a closed loop of ordered edges.

    Parameters:
    - reordered_edge_layer (GeoDataFrame): GeoDataFrame containing the ordered edges forming a loop.

    Returns:
    - Polygon: A Shapely Polygon object representing the closed loop.
    """
    # Extract the geometries from the reordered edge layer
    edge_geometries = list(reordered_edge_layer.geometry)

    # Combine geometries into a single LineString
    combined_line = LineString([point for geom in edge_geometries for point in geom.coords])

    # Ensure the line is closed by checking the first and last points
    if combined_line.coords[0] != combined_line.coords[-1]:
        combined_line = LineString(list(combined_line.coords) + [combined_line.coords[0]])

    # Create a polygon from the closed LineString
    polygon = Polygon(combined_line)

    return polygon

# Example usage:

edge_layer = gpd.read_file("Foundation_Data.gpkg", layer="edges") 
ordered_edge_ids = order_edges_around_island(edge_layer)
reordered_edge_layer = filter_and_reorder_edges(edge_layer, ordered_edge_ids)
polygon = create_polygon_from_edges(reordered_edge_layer)

polygon_gdf = gpd.GeoDataFrame([{"geometry": polygon}], crs=edge_layer.crs)
polygon_gdf.to_file("Jamaica_polygon_revised.gpkg", layer="jam", driver="GPKG")




